# traceability_export — Excel de trazabilidad del cálculo PR (modo B: con fórmulas Excel)

Genera un libro Excel por planta y periodo con **fórmulas Excel reales** en las hojas
calculadas, de modo que el cliente puede modificar umbrales o lecturas y ver el delta
recalcularse en vivo.

**Hojas del libro:**
1. `00_Resumen` — KPI principales (con fórmulas que apuntan a `05_Comparacion`)
2. `01_Config_Contrato` — umbrales aplicados (origen de las referencias de las fórmulas)
3. `02_SCADA_Raw` — lecturas del SCADA tal cual (valores)
4. `03_SCADA_Procesado` — raw + fórmulas (POA filt, ALB, c1-c4, all_valid)
5. `04_PVsyst_Simulacion` — referencia simulada (valores)
6. `05_Comparacion_Interval` — fórmulas: E_medida, E_esperada, E_garantizada, delta
7. `06_Resumen_Diario` — fórmulas SUMIFS por día sobre `05_Comparacion`
9. `08_Outliers_POA` — valores (subset filtrado de `03`)

Las fórmulas usan `LET` y `SUMPRODUCT`, compatibles con Excel 365 / 2021+.

**Período por defecto:** último mes (últimos 30 días desde la fecha máxima en `scada_proc`).


In [1]:
# Parámetros
plant_codes  = "PL2"       # una o varias plantas (lista, JSON o separadas por comas). Vacio = todas
period_start = ""          # ISO YYYY-MM-DD. Si lo rellenas, MANDA sobre last_month_only.
period_end   = ""          # ISO YYYY-MM-DD. Vacío = máximo disponible.


In [2]:


# Filtrar al último mes (últimos 30 días). SOLO se aplica si period_start y period_end están vacíos.
last_month_only = True

# --- App registrada en Azure AD: "PVSyst1_EPI" ---
client_id  = "58f369e5-0dfd-4435-802b-a90a6786bf22"
tenant_id  = "943fa85b-068d-404a-bacb-5f0baf8d31a8"

# Key Vault con el client_secret
key_vault_url             = "https://apicredentialsenerland.vault.azure.net/"
secret_client_secret_name = "PVSyst1EPISecret"

# Destino SharePoint
sp_site_path     = "enerlandgroup.sharepoint.com:/sites/Pvsyst1"
sp_drive_name    = "Documentos"
sp_traceability_path = "TRACEABILITY"

# Audit local
local_audit_root = "traceability_audit"


In [3]:
import io
import json
import requests
import pandas as pd
import numpy as np
import notebookutils
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import quote

LAKEHOUSE_FILES = '/lakehouse/default/Files'
ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read_sheet(sheet, table=None, header=0):
    """Lee una TABLA ESTRUCTURADA del Excel de config: 1) por NOMBRE de tabla
    (case-insensitive, en cualquier hoja); 2) si no, la (primera) tabla de la hoja
    indicada; 3) ultimo recurso, la hoja completa. Robusto ante renombrados."""
    local = '/tmp/plants_config_trace.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    import openpyxl
    wb = openpyxl.load_workbook(local, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(local, sheet_name=sheet, header=header)
def _getf(row, key, default):
    if key not in row.index:
        return default
    v = row[key]
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _getf_v(v, default):
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _bool(v, default=False):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return default
    s = str(v).strip().upper()
    return s in ('TRUE', 'VERDADERO', '1', 'SÍ', 'SI', 'YES', 'Y')


def _parse_selected_criteria(plant_code):
    """04_Criteria_Selection (tabla T04: plant_code, c1..c9). c1-c7 numerico (-1 off), c8-c9 bool."""
    sel = {f'c{i}': (0.0 if i <= 7 else True) for i in range(1, 10)}
    try:
        sc = _read_sheet('04_Criteria_Selection', 'T04_criteria_selection')
        sc.columns = [str(c).strip().lower() for c in sc.columns]
        sub = sc[sc['plant_code'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for n in range(1, 10):
                ck = f'c{n}'
                if ck in sc.columns:
                    sel[ck] = _getf_v(row[ck], 0.0) if n <= 7 else _bool(row[ck], True)
    except Exception:
        pass
    return sel
def _parse_monthly_irr(plant_code):
    """07_Irrandiance_Thresholds (tabla T07: plant_code, GHI_m1..POA_m12)."""
    poa, ghi = {}, {}
    try:
        df = _read_sheet('07_Irrandiance_Thresholds', 'T07_irradiance_threshold')
        df.columns = [str(c).strip().upper() for c in df.columns]
        sub = df[df['PLANT_CODE'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for m in range(1, 13):
                if f'POA_M{m}' in df.columns:
                    poa[m] = _getf_v(row[f'POA_M{m}'], 0.0)
                if f'GHI_M{m}' in df.columns:
                    ghi[m] = _getf_v(row[f'GHI_M{m}'], 0.0)
    except Exception:
        pass
    return poa, ghi
def _load_sensor_outliers(plant_code):
    """05_Sensor_Outliers -> dict {poa,ghi,dhi,ref,tamb} max_dev por sensor."""
    defaults = {'poa': 0.04, 'ghi': 0.0, 'dhi': 0.0, 'ref': 0.0, 'tamb': 0.0}
    try:
        so = _read_sheet('05_Sensor_Outliers', 'T05_sensor_outliers')
        sub = so[so['plant_code'] == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            return {
                'poa':  _getf(row, 'POA_max_dev',  defaults['poa']),
                'ghi':  _getf(row, 'GHI_max_dev',  defaults['ghi']),
                'dhi':  _getf(row, 'DHI_max_dev',  defaults['dhi']),
                'ref':  _getf(row, 'REF_max_dev',  defaults['ref']),
                'tamb': _getf(row, 'TAMB_max_dev', defaults['tamb']),
            }
    except Exception:
        pass
    return defaults


def load_plant_config(plant_code):
    """Lee 01_Plants + 03_Criteria_Config + 04_Criteria_Selection + 07 (+05).
    Devuelve config canonica para la trazabilidad (9 criterios)."""
    plants   = _read_sheet('01_Plants', 'T01_plants')
    criteria = _read_sheet('03_Criteria_Config', 'T03_criteria_config')

    prow = plants[plants['plant_code'] == plant_code]
    if prow.empty:
        raise ValueError(f'{plant_code} not found in 01_Plants')
    prow = prow.iloc[0]

    crow = criteria[criteria['plant_code'] == plant_code]
    if crow.empty:
        raise ValueError(f'{plant_code} not found in 03_Criteria_Config')
    crow = crow.iloc[0]

    sel = _parse_selected_criteria(plant_code)
    poa_m, ghi_m = _parse_monthly_irr(plant_code)

    # Criterios ACTIVOS: c1-c7 activos si valor >= 0; c8-c9 si True.
    active = set()
    for n in range(1, 8):
        if sel.get(f'c{n}', 0.0) is not None and sel.get(f'c{n}', 0.0) >= 0:
            active.add(f'c{n}')
    for n in (8, 9):
        if sel.get(f'c{n}', True):
            active.add(f'c{n}')

    return {
        'plant_code':           plant_code,
        'plant_name':           str(prow['plant_name']),
        'ac_capacity_kw':       _getf(prow, 'ac_capacity_kw', 0.0),
        'dc_capacity_kw':       _getf(prow, 'dc_capacity_kw', 0.0),
        'pvsyst_file':          str(prow.get('pvsyst_file', '')),
        'timezone':             str(prow.get('timezone', '')),

        'guaranteed_epi':       _getf(crow, 'guaranteed_epi', 0.98),
        'degradation_factor':   _getf(crow, 'degradation_factor', 1.0),
        'availability_factor':  _getf(crow, 'availability_factor', 1.0),
        'energy_ref_sensor':    (str(crow.get('energy_ref_sensor', 'E_GRID_01')).strip() or 'E_GRID_01'),
        'albedo_formula':       (lambda _v: _v if _v in ('REF/GHI', 'REF/POA') else 'REF/GHI')(
                                    str(crow.get('albedo_formula', '')).strip().upper().replace(' ', '')),
        # Outlier max_dev por tipo de sensor (05_Sensor_Outliers)
        **dict(zip(
            ['poa_sensor_max_dev','ghi_sensor_max_dev','dhi_sensor_max_dev',
             'ref_sensor_max_dev','tamb_max_dev'],
            [_load_sensor_outliers(plant_code)[k] for k in ('poa','ghi','dhi','ref','tamb')]
        )),

        # --- Criterios (04_Criteria_Selection) ---
        'selected_criteria':    sel,
        'active_criteria':      active,
        # umbrales por criterio (valor de 04)
        'c1_thr':  sel.get('c1', 0.0),
        'c2_thr':  sel.get('c2', 0.0),
        'c3_thr':  sel.get('c3', 0.0),
        'c4_thr':  sel.get('c4', 0.0),
        'c5_hours': sel.get('c5', 0.0),
        'c6_hours': sel.get('c6', 0.0),
        'c7_setpoint': sel.get('c7', 0.0),
        'c8_wind_alarm': bool(sel.get('c8', False)),
        'c9_pf_rango':   bool(sel.get('c9', False)),

        # --- Curvas mensuales de irradiancia minima (07) ---
        'monthly_thr_poa':      poa_m,
        'monthly_thr_ghi':      ghi_m,

        'aggregate_by_hour':    _bool(crow.get('aggregate_by_hour') if 'aggregate_by_hour' in crow.index else None, False),
        'is_energy':            _bool(crow.get('is_energy') if 'is_energy' in crow.index else None, False),
    }

In [4]:
# Microsoft Graph helpers (mismo flow que meteo_export / pr_runner)
_GRAPH_BASE = "https://graph.microsoft.com/v1.0"
_TIMEOUT    = 60


def get_graph_token():
    client_secret = notebookutils.credentials.getSecret(key_vault_url, secret_client_secret_name)
    r = requests.post(
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        data={'client_id': client_id, 'scope': 'https://graph.microsoft.com/.default',
              'client_secret': client_secret, 'grant_type': 'client_credentials'},
        timeout=_TIMEOUT,
    )
    if not r.ok:
        raise RuntimeError(f"OAuth fallo {r.status_code}: {r.text[:400]}")
    return r.json()['access_token']


def _gget(token, path):
    r = requests.get(f"{_GRAPH_BASE}/{path.lstrip('/')}",
                     headers={'Authorization': f'Bearer {token}'},
                     timeout=_TIMEOUT)
    if not r.ok:
        raise RuntimeError(f"Graph GET {path} -> {r.status_code}: {r.text[:400]}")
    return r.json()


def resolve_site_and_drive(token):
    site = _gget(token, f"sites/{sp_site_path}")
    site_id = site['id']
    if sp_drive_name:
        drives = _gget(token, f"sites/{site_id}/drives").get('value', [])
        drive = next((d for d in drives if d.get('name') == sp_drive_name), None)
        if drive is None:
            raise ValueError(f"Biblioteca '{sp_drive_name}' no encontrada. Disponibles: {[d.get('name') for d in drives]}")
        drive_id = drive['id']
    else:
        drive_id = _gget(token, f"sites/{site_id}/drive")['id']
    return site_id, drive_id


def upload_to_sharepoint(token, site_id, drive_id, dest_relpath, content_bytes):
    safe_path = "/".join(quote(seg, safe='') for seg in dest_relpath.strip('/').split('/'))
    url = f"{_GRAPH_BASE}/sites/{site_id}/drives/{drive_id}/root:/{safe_path}:/content"
    r = requests.put(
        url,
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/octet-stream'},
        data=content_bytes,
        timeout=_TIMEOUT * 4,
    )
    if not r.ok:
        raise RuntimeError(f"Graph PUT {dest_relpath} -> {r.status_code}: {r.text[:400]}")
    j = r.json()
    return {'name': j.get('name'), 'webUrl': j.get('webUrl'), 'size': j.get('size')}


In [5]:
# ---------------------------------------------------------------------------
# Excel builder (openpyxl) - modo hibrido:
#   - Hojas 02/03/04/08: VALORES precomputados de las tablas Delta (sin formulas).
#   - Hojas 05/06/07/00: FORMULAS Excel que se recalculan al cambiar EPI/Deg/Avail.
#   - Hoja 01: configuracion (origen de referencias para las formulas).
# ---------------------------------------------------------------------------
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# --- Estilos ---------------------------------------------------------------
_HDR_FILL    = PatternFill('solid', start_color='1F4E79', end_color='1F4E79')
_HDR_FONT    = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
_HDR_ALIGN   = Alignment(horizontal='center', vertical='center', wrap_text=True)
_NOTE_FILL   = PatternFill('solid', start_color='FFF2CC', end_color='FFF2CC')
_NOTE_FONT   = Font(name='Calibri', size=10, italic=True, color='3D3D3D')
_FORMULA_FILL= PatternFill('solid', start_color='F2F2F2', end_color='F2F2F2')
_KPI_FILL_OK = PatternFill('solid', start_color='E2EFDA', end_color='E2EFDA')
_KPI_FILL_KO = PatternFill('solid', start_color='FCE4D6', end_color='FCE4D6')
_KPI_FONT    = Font(name='Calibri', size=12, bold=True)
_THIN        = Side(style='thin', color='BFBFBF')
_BORDER      = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)

# Colores de estado / anomalia
_NULL_FILL      = PatternFill('solid', start_color='D9D9D9', end_color='D9D9D9')  # gris
_ANOMALY_FILL   = PatternFill('solid', start_color='F4B6B6', end_color='F4B6B6')  # rojo claro
_OUTLIER_FILL   = PatternFill('solid', start_color='F8CBAD', end_color='F8CBAD')  # salmon
_DISCARDED_FILL = PatternFill('solid', start_color='FFE699', end_color='FFE699')  # ambar
_STATUS_FONT    = Font(name='Calibri', size=10, italic=True, bold=True, color='843C0C')
_STATUS_ALIGN   = Alignment(horizontal='center', vertical='center')

# --- Referencias absolutas a 01_Config_Contrato (filas hardcoded) ----------
CFG = "'01_Config_Contrato'"
# Filas FIJAS del layout de 01_Config_Contrato (ver _config_layout). NO reordenar.
REF_EPI_GUAR     = f"{CFG}!$B$8"   # guaranteed_epi
REF_DEGRADATION  = f"{CFG}!$B$9"   # degradation_factor
REF_AVAILABILITY = f"{CFG}!$B$10"  # availability_factor
REF_C1_THR       = f"{CFG}!$B$16"  # c1 umbral POA_avg (0 = solo no-null)
REF_C2_THR       = f"{CFG}!$B$17"  # c2 umbral GHI_avg
REF_C3_THR       = f"{CFG}!$B$18"  # c3 umbral DHI_avg
REF_C4_THR       = f"{CFG}!$B$19"  # c4 umbral REF_avg
REF_C5_HOURS     = f"{CFG}!$B$20"  # c5 nº horas/dia POA
REF_C6_HOURS     = f"{CFG}!$B$21"  # c6 nº horas/dia GHI
REF_C7_SETPT     = f"{CFG}!$B$22"  # c7 umbral setpoint kW
REF_POA_DEV      = f"{CFG}!$B$26"  # poa_sensor_max_dev  (05_Sensor_Outliers, row 26)
REF_GHI_DEV      = f"{CFG}!$B$27"  # ghi_sensor_max_dev  (05_Sensor_Outliers, row 27)
REF_DHI_DEV      = f"{CFG}!$B$28"  # dhi_sensor_max_dev  (05_Sensor_Outliers, row 28)
REF_REF_DEV      = f"{CFG}!$B$29"  # ref_sensor_max_dev  (05_Sensor_Outliers, row 29)
REF_TAMB_DEV     = f"{CFG}!$B$30"  # tamb_max_dev        (05_Sensor_Outliers, row 30)
REF_DT_H         = f"{CFG}!$B$32"  # dt_h (row 32)


def _explain(ws, text, ncols=8, row=1):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    c = ws.cell(row=row, column=1, value=text)
    c.fill = _NOTE_FILL
    c.font = _NOTE_FONT
    c.alignment = Alignment(wrap_text=True, vertical='top', horizontal='left')
    n_lines = max(text.count('\n') + 1, 3)
    ws.row_dimensions[row].height = 18 * n_lines


def _hdr(ws, cols, row):
    for i, c in enumerate(cols, 1):
        cell = ws.cell(row=row, column=i, value=c)
        cell.fill = _HDR_FILL
        cell.font = _HDR_FONT
        cell.alignment = _HDR_ALIGN
        cell.border = _BORDER
    ws.row_dimensions[row].height = 28


def _mark_status(cell, label):
    upper = (label or '').upper()
    fill_map = {
        'NULL':         _NULL_FILL,
        'MISSING':      _NULL_FILL,
        'ANOMALY':      _ANOMALY_FILL,
        'OUT_OF_RANGE': _ANOMALY_FILL,
        'DEAD':         _ANOMALY_FILL,
        'ABRUPT':       _ANOMALY_FILL,
        'OUTLIER':      _OUTLIER_FILL,
        'DISCARDED':    _DISCARDED_FILL,
    }
    fill = fill_map.get(upper)
    if fill:
        cell.fill = fill
    cell.font = _STATUS_FONT
    cell.alignment = _STATUS_ALIGN


def _apply_status_cf(ws, top_left, bottom_right):
    """Aplica formato condicional a un rango: pinta de color las celdas que contienen
    exactamente 'NULL'/'MISSING'/'ANOMALY'/.../'OUTLIER'/'DISCARDED'.
    """
    from openpyxl.formatting.rule import CellIsRule
    rng = f'{top_left}:{bottom_right}'
    rules = [
        ('NULL',         _NULL_FILL),
        ('MISSING',      _NULL_FILL),
        ('ANOMALY',      _ANOMALY_FILL),
        ('OUT_OF_RANGE', _ANOMALY_FILL),
        ('DEAD',         _ANOMALY_FILL),
        ('ABRUPT',       _ANOMALY_FILL),
        ('OUTLIER',      _OUTLIER_FILL),
        ('DISCARDED',    _DISCARDED_FILL),
    ]
    for label, fill in rules:
        ws.conditional_formatting.add(rng,
            CellIsRule(operator='equal', formula=[f'"{label}"'], fill=fill, font=_STATUS_FONT))


def _put_val(ws, row, col, v, fmt=None):
    """Escribe un valor en una celda.
       - Empty string ('') -> deja la celda VACIA (sin marcar nada). Usado para separadores
         visuales en 01_Config_Contrato.
       - None / NaN / Inf -> escribe 'NULL' con fondo gris.
       - Strings de estado conocidos ('OUTLIER', 'DISCARDED', 'OUT_OF_RANGE', etc.) ->
         coloreados con _mark_status.
       - 'ok' -> texto gris suave (no resalta).
       - Resto -> valor tal cual con formato opcional.
    """
    # Empty string -> dejar la celda vacia sin marcar
    if isinstance(v, str) and not v.strip():
        return

    # None / NaN / Inf -> NULL coloreado
    is_null = v is None or (isinstance(v, float) and not np.isfinite(v))
    try:
        if not is_null and pd.isna(v):
            is_null = True
    except (TypeError, ValueError):
        pass

    if is_null:
        cell = ws.cell(row=row, column=col, value='NULL')
        _mark_status(cell, 'NULL')
        return

    if isinstance(v, (pd.Timestamp, datetime)):
        cell = ws.cell(row=row, column=col, value=v.to_pydatetime() if hasattr(v, 'to_pydatetime') else v)
        cell.number_format = fmt or 'yyyy-mm-dd hh:mm'
        return

    if isinstance(v, bool):
        ws.cell(row=row, column=col, value=int(v))
        return

    if isinstance(v, (int, np.integer)):
        cell = ws.cell(row=row, column=col, value=int(v))
        if fmt:
            cell.number_format = fmt
        return

    if isinstance(v, (float, np.floating)):
        cell = ws.cell(row=row, column=col, value=float(v))
        if fmt:
            cell.number_format = fmt
        return

    s = str(v)
    cell = ws.cell(row=row, column=col, value=s)
    upper = s.upper()
    if upper in ('NULL', 'OUTLIER', 'DISCARDED', 'ANOMALY',
                 'OUT_OF_RANGE', 'MISSING', 'DEAD', 'ABRUPT'):
        _mark_status(cell, upper)
    elif upper == 'OK':
        # Sensor sano: texto gris italica suave (no resalta)
        cell.font = Font(name='Calibri', size=10, color='6F6F6F', italic=True)
        cell.alignment = _STATUS_ALIGN



def _is_null_value(v):
    """True si v es None / NaN / Inf / pd.NA."""
    if v is None:
        return True
    if isinstance(v, float) and not np.isfinite(v):
        return True
    try:
        if pd.isna(v):
            return True
    except (TypeError, ValueError):
        pass
    return False


def _flag_to_label(flag_v):
    """Traduce un valor de columna *_flag a la etiqueta de display.
    Devuelve None si el flag indica que el sensor esta ok / vacio.
    """
    if flag_v is None:
        return None
    try:
        if pd.isna(flag_v):
            return None
    except (TypeError, ValueError):
        pass
    s = str(flag_v).strip().lower()
    if s in ('', 'ok', 'nan', 'none'):
        return None
    return s.upper()


def _put_raw(ws, row, col, raw_v, flag_v=None, fmt=None):
    """Para columnas raw de sensor: si el flag indica anomalia, escribe la
    etiqueta del flag (ANOMALY_X) en lugar de NULL.
    """
    if _is_null_value(raw_v):
        label = _flag_to_label(flag_v)
        if label:
            cell = ws.cell(row=row, column=col, value=label)
            _mark_status(cell, label)
        else:
            cell = ws.cell(row=row, column=col, value='NULL')
            _mark_status(cell, 'NULL')
        return
    _put_val(ws, row, col, raw_v, fmt=fmt)


def _put_filt(ws, row, col, filt_v, raw_v, flag_v=None, fmt=None):
    """Para columnas <sensor>_filt:
       - Si filt es NaN y el flag indica anomalia -> etiqueta del flag.
       - Si filt es NaN y raw tambien NaN -> NULL (sensor sin reporte).
       - Si filt es NaN pero raw era numerico -> DISCARDED (cross-sensor filter).
    """
    if _is_null_value(filt_v):
        label = _flag_to_label(flag_v)
        if label:
            cell = ws.cell(row=row, column=col, value=label)
            _mark_status(cell, label)
        elif _is_null_value(raw_v):
            cell = ws.cell(row=row, column=col, value='NULL')
            _mark_status(cell, 'NULL')
        else:
            cell = ws.cell(row=row, column=col, value='DISCARDED')
            _mark_status(cell, 'DISCARDED')
        return
    _put_val(ws, row, col, filt_v, fmt=fmt)


def _put_avg_filt(ws, row, col, avg_v, family_raw_values, fmt=None):
    """Para columnas <FAMILY>_avg_filt: si NaN, distingue NULL (todos los sensores
    de la familia eran NaN) vs DISCARDED (al menos un sensor era numerico pero el
    filtro lo descarto).
    """
    if _is_null_value(avg_v):
        all_null = all(_is_null_value(v) for v in family_raw_values)
        label = 'NULL' if all_null else 'DISCARDED'
        cell = ws.cell(row=row, column=col, value=label)
        _mark_status(cell, label)
        return
    _put_val(ws, row, col, avg_v, fmt=fmt)


def _autosize(ws, sample_rows=200, min_w=10, max_w=38):
    for col_cells in ws.columns:
        first = next((c for c in col_cells if hasattr(c, 'column_letter')), None)
        if first is None:
            continue
        letter = first.column_letter
        max_len = min_w
        for cell in list(col_cells)[:sample_rows]:
            try:
                if cell.value is None:
                    continue
                s = str(cell.value)
                if len(s) > max_len:
                    max_len = len(s)
            except Exception:
                pass
        ws.column_dimensions[letter].width = min(max_w, max_len + 2)


def _add_table(ws, name, header_row, last_data_row, n_cols):
    """Convierte un rango (header_row..last_data_row) x (1..n_cols) en una Tabla Excel.
    El nombre 'name' es el identificador para referencias estructuradas (p.ej. tbl_proc).
    """
    if last_data_row < header_row or n_cols < 1:
        return
    ref = f"A{header_row}:{get_column_letter(n_cols)}{last_data_row}"
    tab = Table(displayName=name, ref=ref)
    tab.tableStyleInfo = TableStyleInfo(name="TableStyleLight9",
                                        showFirstColumn=False, showLastColumn=False,
                                        showRowStripes=True, showColumnStripes=False)
    ws.add_table(tab)



def _config_layout(cfg, dt_h):
    """Layout FIJO de 01_Config_Contrato. Las constantes REF_* dependen de estas
    filas — NO REORDENAR. Filas 16-22 = umbrales de criterios (04_Criteria_Selection)."""
    sel = cfg.get('selected_criteria', {})
    rows = [
        ('--- Planta ---', ''),                                              # 3
        ('plant_code',           cfg['plant_code']),                         # 4
        ('plant_name',           cfg['plant_name']),                         # 5
        ('ac_capacity_kw',       cfg['ac_capacity_kw']),                     # 6
        ('dc_capacity_kw',       cfg['dc_capacity_kw']),                     # 7
        ('guaranteed_epi',       cfg['guaranteed_epi']),                     # 8  -> REF_EPI_GUAR
        ('degradation_factor',   cfg.get('degradation_factor', 1.0)),        # 9  -> REF_DEGRADATION
        ('availability_factor',  cfg.get('availability_factor', 1.0)),       # 10 -> REF_AVAILABILITY
        ('energy_ref_sensor',    cfg.get('energy_ref_sensor', 'E_GRID_01')), # 11
        ('pvsyst_file',          cfg.get('pvsyst_file', '')),                # 12
        ('timezone',             cfg.get('timezone', '')),                   # 13
        ('', ''),                                                            # 14
        ('--- Criterios (04; -1=desactivado | c8-c9: TRUE/FALSE) ---', ''),  # 15
        ('c1 POA_avg umbral (0=no-null)', sel.get('c1', 0.0)),               # 16 -> REF_C1_THR
        ('c2 GHI_avg umbral (0=no-null)', sel.get('c2', 0.0)),               # 17 -> REF_C2_THR
        ('c3 DHI_avg umbral (0=no-null)', sel.get('c3', 0.0)),               # 18 -> REF_C3_THR
        ('c4 REF_avg umbral (0=no-null)', sel.get('c4', 0.0)),               # 19 -> REF_C4_THR
        ('c5 horas/dia POA',     sel.get('c5', 0.0)),                        # 20 -> REF_C5_HOURS
        ('c6 horas/dia GHI',     sel.get('c6', 0.0)),                        # 21 -> REF_C6_HOURS
        ('c7 setpoint min kW',   sel.get('c7', 0.0)),                              # 22 -> REF_C7_SETPT
        ('c8 Wind Alarm activa', bool(sel.get('c8', False))),                      # 23 (bool, sin formula)
        ('c9 PF fuera de rango', bool(sel.get('c9', False))),                      # 24 (bool, sin formula)
        ('--- Deteccion outliers por sensor (05_Sensor_Outliers) ---', ''),        # 25
        ('poa_sensor_max_dev',   cfg.get('poa_sensor_max_dev', 0.04)),             # 26 -> REF_POA_DEV
        ('ghi_sensor_max_dev',   cfg.get('ghi_sensor_max_dev', 0.0)),              # 27 -> REF_GHI_DEV
        ('dhi_sensor_max_dev',   cfg.get('dhi_sensor_max_dev', 0.0)),              # 28 -> REF_DHI_DEV
        ('ref_sensor_max_dev',   cfg.get('ref_sensor_max_dev', 0.0)),              # 29 -> REF_REF_DEV
        ('tamb_max_dev',         cfg.get('tamb_max_dev', 0.0)),                    # 30 -> REF_TAMB_DEV
        ('', ''),                                                                  # 31
        ('dt_h (paso horario)',  dt_h),                                            # 32 -> REF_DT_H
    ]
    # Curvas mensuales de irradiancia minima (07) si c5 o c6 estan activos.
    act = cfg.get('active_criteria', set())
    if 'c5' in act or 'c6' in act:
        poa_m = cfg.get('monthly_thr_poa', {})
        ghi_m = cfg.get('monthly_thr_ghi', {})
        rows.append(('', ''))
        rows.append(('--- Curvas mensuales minimo irradiancia (07) ---', ''))
        rows.append(('mes',          'POA_min / GHI_min'))
        for m in range(1, 13):
            rows.append((f'  mes {m:02d}', f"{poa_m.get(m, 0.0):g} / {ghi_m.get(m, 0.0):g}"))
    return rows


def _order_proc_columns(cols_in, df=None, active=None):
    """Ordena las columnas de scada_proc de forma legible en la hoja 03."""
    if df is not None:
        cols_in = [c for c in cols_in if c not in df.columns or not df[c].isna().all()]
    skip = {'plant_code', 'source_file', 'ingested_at', 'processed_at'}
    # Las columnas *_flag no se muestran como columna propia (el motivo se anota en
    # la celda _filt); siguen en la tabla y se usan para esa anotacion.
    cols_in = [c for c in cols_in if c not in skip and not c.endswith('_flag')]

    def _related(base):
        out = [base]
        for suf in ('_filt',):
            n = f'{base}{suf}'
            if n in cols_in:
                out.append(n)
        return out

    used = set()
    ordered = []
    if 'Date' in cols_in:
        ordered.append('Date'); used.add('Date')

    def _add_family(family, avg_filt_name=None):
        bases = sorted([c for c in cols_in
                        if c.upper().startswith(family)
                        and not c.endswith('_filt') and not c.endswith('_flag')
                        and c != avg_filt_name])
        for base in bases:
            for x in _related(base):
                if x not in used:
                    ordered.append(x); used.add(x)
        if avg_filt_name and avg_filt_name in cols_in and avg_filt_name not in used:
            ordered.append(avg_filt_name); used.add(avg_filt_name)

    _add_family('GHI_', 'GHI_avg_filt')
    _add_family('POA_', 'POA_avg_filt')
    _add_family('REF_', 'REF_avg_filt')
    _add_family('DHI_', 'DHI_avg_filt')
    _add_family('T_AMB', 'T_AMB_avg_filt')
    _add_family('WS_')
    _add_family('SETPOINT')
    _add_family('E_GRID')

    # Albedo
    for base in sorted([c for c in cols_in if c.upper().startswith('ALB_') and c != 'ALB_avg_filt']):
        if base not in used:
            ordered.append(base); used.add(base)
    if 'ALB_avg_filt' in cols_in and 'ALB_avg_filt' not in used:
        ordered.append('ALB_avg_filt'); used.add('ALB_avg_filt')

    _add_family('WA')
    _add_family('PF_')
    # Criterios y derivados. Solo se muestran los criterios ACTIVOS (active).
    _all_crit = ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
    crit_seq = _all_crit if active is None else [c for c in _all_crit if c in active]
    derived = ['poa_above_thr']
    if active is None or 'c5' in active:
        derived.append('daily_hours_above')
    if active is None or 'c6' in active:
        derived.append('daily_hours_above_ghi')
    derived.append('hours_aggregated')
    # Criterios INACTIVOS (y sus horas/dia) NO deben mostrarse ni entrar en all_valid.
    _skip = set(_all_crit) - set(crit_seq)
    if active is not None:
        if 'c5' not in active:
            _skip.add('daily_hours_above')
        if 'c6' not in active:
            _skip.add('daily_hours_above_ghi')
    for x in derived + crit_seq + ['all_valid']:
        if x in cols_in and x not in used and x not in _skip:
            ordered.append(x); used.add(x)

    for c in cols_in:
        if c not in used and c not in _skip:
            ordered.append(c); used.add(c)
    return ordered


def build_workbook(plant_code, period_str, cfg, dt_h,
                   scada_raw_df, scada_proc_df, pvsyst_df, outliers_df):
    """Modo hibrido:
        02/03/04/08 -> valores precomputados (scada_raw, scada_proc, proc_pvsyst).
        05/06/07/00 -> formulas Excel sobre las celdas anteriores (recalculan en vivo).
    """
    wb = Workbook()
    wb.remove(wb.active)

    # ============================================================
    # 01_Config_Contrato
    # ============================================================
    ws_cfg = wb.create_sheet('01_Config_Contrato')
    _explain(ws_cfg,
        "Parametros leidos de plants_config.xlsx (01_Plants y 03_Criteria) y aplicados al calculo del EPI. "
        "Las celdas de columna B son referencias usadas por las formulas en 05_Comparacion_Interval, "
        "06_Resumen_Diario y 00_Resumen. Modifica un valor y todo lo dependiente "
        "se recalcula en vivo.",
        ncols=2)
    for i, (k, v) in enumerate(_config_layout(cfg, dt_h)):
        ws_cfg.cell(row=3 + i, column=1, value=k).font = Font(bold=k.startswith('---'))
        _put_val(ws_cfg, 3 + i, 2, v)
    ws_cfg.column_dimensions['A'].width = 32
    ws_cfg.column_dimensions['B'].width = 24

    # ============================================================
    # 00_Resumen (insertado primero; rellenado al final)
    # ============================================================
    ws_sum = wb.create_sheet('00_Resumen', 0)
    ws_sum.merge_cells('A1:F1')
    title = ws_sum.cell(row=1, column=1, value="Performance Ratio - Trazabilidad")
    title.font = Font(name='Calibri', size=18, bold=True, color='1F4E79')
    title.alignment = Alignment(horizontal='left', vertical='center')
    ws_sum.row_dimensions[1].height = 32

    # ============================================================
    # 02_SCADA_Raw (valores tal cual)
    # ============================================================
    ws_raw = wb.create_sheet('02_SCADA_Raw')
    _explain(ws_raw,
        "Lecturas del SCADA tal cual llegaron del operador, renombradas a la nomenclatura estandar "
        "(GHI_01, POA_01, REF_01, T_AMB_01, E_GRID_01, SETPOINT_01, WS_01) segun "
        "plants_config.02_Sensors_Mapping. Sin transformaciones.",
        ncols=min(len(scada_raw_df.columns), 12))
    raw_cols = [c for c in scada_raw_df.columns if not scada_raw_df[c].isna().all()]
    scada_raw_df = scada_raw_df[raw_cols]  
    _hdr(ws_raw, raw_cols, 3)
    for r_idx, row_vals in enumerate(scada_raw_df.itertuples(index=False, name=None), start=4):
        for c_idx, v in enumerate(row_vals, 1):
            _put_val(ws_raw, r_idx, c_idx, v,
                     fmt=('yyyy-mm-dd hh:mm' if raw_cols[c_idx-1] == 'Date' else None))
    ws_raw.freeze_panes = 'B4'
    _autosize(ws_raw)
    if len(scada_raw_df) > 0:
        _apply_status_cf(ws_raw, 'A4', f'{get_column_letter(len(raw_cols))}{4 + len(scada_raw_df) - 1}')
        _add_table(ws_raw, 'tbl_raw', 3, 4 + len(scada_raw_df) - 1, len(raw_cols))

    # ============================================================
    # 03_SCADA_Procesado (valores de scada_proc, precomputados)
    # ============================================================
    ws_proc = wb.create_sheet('03_SCADA_Procesado')
    _explain(ws_proc,
        "Datos del SCADA tras aplicar el pipeline en scada_proc.Notebook:\n"
        " 1. Anomaly detection per sensor (out_of_range / missing / dead / abrupt) -> celda coloreada.\n"
        " 2. Cross-sensor outlier en POA (4%). Si la planta lo configura tambien GHI/DHI/T_amb (Plaza II: off).\n"
        " 3. Albedo = REF / base (GHI o POA segun 03_Criteria.albedo_formula); un ALB por subindice.\n"
        " 4. Criterios activos (04_Criteria_Selection): c1-c4 salud avg_filt y c7-c9 (formulas vivas);\n    c5/c6 horas/dia validas POA/GHI (valores de scada_proc). all_valid = producto de los activos.\n"
        "Las celdas con etiqueta 'NULL' (gris), 'OUT_OF_RANGE/DEAD/ABRUPT' (rojo), 'OUTLIER' (salmon) o "
        "'DISCARDED' (ambar) indican que ese valor se descarto en el filtro indicado.",
        ncols=14)

    headers = _order_proc_columns(list(scada_proc_df.columns), df=scada_proc_df,
                                  active=cfg.get('active_criteria'))
    _hdr(ws_proc, headers, 3)
    col_letter = {h: get_column_letter(i+1) for i, h in enumerate(headers)}

    data_start = 4
    n_rows = len(scada_proc_df)
    data_end = data_start + n_rows - 1

    proc_cols_in = list(scada_proc_df.columns)
    # Mapas para distinguir NULL vs DISCARDED al volcar valores
    # 1) <sensor>_filt -> raw base (POA_01_filt -> POA_01)
    # Mapas para distinguir NULL / DISCARDED / ANOMALY al volcar valores
    filt_to_raw = {}     # 'POA_01_filt' -> 'POA_01'
    raw_to_flag = {}     # 'POA_01' -> 'POA_01_flag'
    for h in headers:
        if h.endswith('_filt') and not h.endswith('_avg_filt'):
            base = h[:-len('_filt')]
            if base in proc_cols_in:
                filt_to_raw[h] = base
    # _flag ya no es columna visible: se localiza en la tabla completa para anotar
    # las celdas raw/_filt con el motivo (dead/missing/out_of_range/abrupt).
    for c in proc_cols_in:
        if c.endswith('_flag'):
            base = c[:-len('_flag')]
            if base in proc_cols_in:
                raw_to_flag[base] = c

    avg_filt_to_family = {
        'POA_avg_filt':   'POA_',
        'GHI_avg_filt':   'GHI_',
        'DHI_avg_filt':   'DHI_',
        'T_AMB_avg_filt': 'T_AMB',
        'ALB_avg_filt':   'ALB_',
    }
    avg_filt_to_raws = {}
    for avg_col, prefix in avg_filt_to_family.items():
        if avg_col not in proc_cols_in:
            continue
        raws = [c for c in proc_cols_in
                if c.upper().startswith(prefix.upper())
                and not c.endswith('_filt')
                and not c.endswith('_flag')
                and c != avg_col]
        avg_filt_to_raws[avg_col] = raws

    # avg_filt como FORMULA = PROMEDIO de las columnas *_filt de la familia (vinculado).
    # (ALB_avg_filt se trata aparte mas abajo.)
    avg_filt_to_filt = {}
    for avg_col, prefix in avg_filt_to_family.items():
        if avg_col not in headers or avg_col == 'ALB_avg_filt':
            continue
        fcols = [c for c in headers
                 if c.upper().startswith(prefix.upper())
                 and c.endswith('_filt') and c != avg_col]
        if fcols:
            avg_filt_to_filt[avg_col] = fcols

    # Pre-computar letras de columna que las formulas necesitan
    ghi_col_letter      = next((col_letter[c] for c in headers
                                if c.upper().startswith('GHI_')
                                and not c.endswith('_filt')
                                and not c.endswith('_flag')), None)
    setpoint_col_letter = next((col_letter[c] for c in headers
                                if c.upper().startswith('SETPOINT')
                                and not c.endswith('_flag')), None)
    ws_col_letter       = next((col_letter[c] for c in headers
                                if c.upper().startswith('WS_')
                                and not c.endswith('_flag')), None)
    poa_avg_letter      = col_letter.get('POA_avg_filt')
    ghi_avg_letter      = col_letter.get('GHI_avg_filt')
    dhi_avg_letter      = col_letter.get('DHI_avg_filt')
    ref_avg_letter      = col_letter.get('REF_avg_filt')
    daily_hours_letter  = col_letter.get('daily_hours_above')
    ghi_daily_letter    = col_letter.get('daily_hours_above_ghi')
    poa_above_letter    = col_letter.get('poa_above_thr')
    wa_col_letter       = next((col_letter[c] for c in headers
                                if (c.upper().startswith('WA_') or c.upper() == 'WA')
                                and not c.endswith('_flag') and not c.endswith('_filt')), None)
    pf_col_letter       = next((col_letter[c] for c in headers
                                if c.upper().startswith('PF_')
                                and not c.endswith('_flag') and not c.endswith('_filt')), None)

    # Mapa ALB_XX -> (GHI_col, REF_col) emparejado por sufijo de estacion
    def _sfx_col(name):
        return name.split('_')[-1] if '_' in name else ''
    _alb_base = 'POA' if str(cfg.get('albedo_formula', 'REF/GHI')).strip().upper().replace(' ', '').endswith('POA') else 'GHI'
    _base_in_headers = [c for c in headers
                        if c.upper().startswith(f'{_alb_base}_')
                        and not c.endswith('_filt') and not c.endswith('_flag')]
    _ref_in_headers = [c for c in headers
                       if c.upper().startswith('REF_')
                       and not c.endswith('_filt') and not c.endswith('_flag')]
    alb_to_pair = {}
    for _bc in _base_in_headers:
        _sfx = _sfx_col(_bc)
        _rc = next((r for r in _ref_in_headers if _sfx_col(r) == _sfx), None)
        if _rc:
            _alb = f'ALB_{_sfx}'
            if _alb in headers:
                alb_to_pair[_alb] = (_bc, _rc)
    alb_cols_in_headers = sorted([c for c in headers
                                  if c.upper().startswith('ALB_') and c != 'ALB_avg_filt'])

    for r_offset, row_vals in enumerate(scada_proc_df.itertuples(index=False, name=None)):
        r = data_start + r_offset
        for h in headers:
            if h not in proc_cols_in:
                continue
            idx = proc_cols_in.index(h)
            v = row_vals[idx]
            col_idx = headers.index(h) + 1
            fmt = 'yyyy-mm-dd hh:mm' if h == 'Date' else None

            # ---- Criterios como FORMULAS (no valores) ---------------------
            # ---- ALB_XX como FORMULA: REF/base (config) con convencion Plaza II ----
            #      den NaN o REF NaN -> "" ; den==0 o ratio<0 -> 0 ; resto -> REF/den
            if h in alb_to_pair:
                _den_col, _ref_col = alb_to_pair[h]
                _den_letter = col_letter[_den_col]
                _ref_letter = col_letter[_ref_col]
                f = (f'=IFERROR(IF(OR(NOT(ISNUMBER({_den_letter}{r})),NOT(ISNUMBER({_ref_letter}{r}))),"",'
                     f'IF({_den_letter}{r}=0,0,IF({_ref_letter}{r}/{_den_letter}{r}<0,0,'
                     f'{_ref_letter}{r}/{_den_letter}{r}))),"")')
                c = ws_proc.cell(row=r, column=col_idx, value=f)
                c.fill = _FORMULA_FILL
                c.number_format = '0.000'
                continue

            if h == 'ALB_avg_filt' and alb_cols_in_headers:
                # Media de ALB_XX en [0, 1). Vacios o fuera de rango se ignoran.
                _first = col_letter[alb_cols_in_headers[0]]
                _last  = col_letter[alb_cols_in_headers[-1]]
                _alb_range = f'{_first}{r}:{_last}{r}'
                f = f'=IFERROR(AVERAGEIFS({_alb_range},{_alb_range},">0",{_alb_range},"<1"),0)'
                c = ws_proc.cell(row=r, column=col_idx, value=f)
                c.fill = _FORMULA_FILL
                c.number_format = '0.000'
                continue

            # ---- Criterios c1-c4 (salud avg_filt) y c7-c9: FORMULAS VIVAS ----
            #      (c5/c6 NO son formula: se vuelcan los valores de scada_proc)
            if h == 'c1':
                f = (f'=IF(ISNUMBER({poa_avg_letter}{r}),IF({REF_C1_THR}<=0,1,'
                     f'IF({poa_avg_letter}{r}>={REF_C1_THR},1,0)),0)') if poa_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c2':
                f = (f'=IF(ISNUMBER({ghi_avg_letter}{r}),IF({REF_C2_THR}<=0,1,'
                     f'IF({ghi_avg_letter}{r}>={REF_C2_THR},1,0)),0)') if ghi_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c3':
                f = (f'=IF(ISNUMBER({dhi_avg_letter}{r}),IF({REF_C3_THR}<=0,1,'
                     f'IF({dhi_avg_letter}{r}>={REF_C3_THR},1,0)),0)') if dhi_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c4':
                f = (f'=IF(ISNUMBER({ref_avg_letter}{r}),IF({REF_C4_THR}<=0,1,'
                     f'IF({ref_avg_letter}{r}>={REF_C4_THR},1,0)),0)') if ref_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c5':
                c_obj = ws_proc.cell(row=r, column=col_idx, value=v)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c6':
                c_obj = ws_proc.cell(row=r, column=col_idx, value=v)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c7':
                # setpoint >= umbral -> el operador NO limita la potencia
                f = (f'=IF(AND(ISNUMBER({setpoint_col_letter}{r}),'
                     f'{setpoint_col_letter}{r}>={REF_C7_SETPT}),1,0)') if setpoint_col_letter else '=1'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c8':
                # sin Wind Alarm (WA != 1)
                f = f'=IF(N({wa_col_letter}{r})=1,0,1)' if wa_col_letter else '=1'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c9':
                # PF sano (0.98 < PF < 1.0)
                f = (f'=IF(AND(ISNUMBER({pf_col_letter}{r}),{pf_col_letter}{r}>0.98,'
                     f'{pf_col_letter}{r}<1),1,0)') if pf_col_letter else '=1'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue

            if h == 'all_valid':
                # producto de los criterios ACTIVOS presentes en la hoja
                _present = [col_letter.get(ck) for ck in
                            ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
                            if col_letter.get(ck)]
                f = ('=' + '*'.join(f'{l}{r}' for l in _present)) if _present else '=1'
                ws_proc.cell(row=r, column=col_idx, value=f).fill = _FORMULA_FILL
                continue

            if h in avg_filt_to_filt:
                # PROMEDIO de los *_filt de la familia (ignora texto NULL/DISCARDED).
                _cells = ','.join(f'{col_letter[fc]}{r}' for fc in avg_filt_to_filt[h])
                fobj = ws_proc.cell(row=r, column=col_idx, value=f'=IFERROR(AVERAGE({_cells}),"")')
                fobj.fill = _FORMULA_FILL
                fobj.number_format = '0.00'
                continue

            # ---- Volcado normal de valores --------------------------------
            if h in filt_to_raw:
                base = filt_to_raw[h]
                raw_v = row_vals[proc_cols_in.index(base)]
                flag_v = row_vals[proc_cols_in.index(raw_to_flag[base])] if base in raw_to_flag else None
                _put_filt(ws_proc, r, col_idx, v, raw_v, flag_v=flag_v, fmt=fmt)
            elif h in avg_filt_to_raws:
                family_raws = [row_vals[proc_cols_in.index(c)] for c in avg_filt_to_raws[h]]
                _put_avg_filt(ws_proc, r, col_idx, v, family_raws, fmt=fmt)
            elif h in raw_to_flag:
                flag_v = row_vals[proc_cols_in.index(raw_to_flag[h])]
                _put_raw(ws_proc, r, col_idx, v, flag_v=flag_v, fmt=fmt)
            else:
                _put_val(ws_proc, r, col_idx, v, fmt=fmt)
    ws_proc.freeze_panes = 'B4'
    _autosize(ws_proc, max_w=18)
    if n_rows > 0:
        _apply_status_cf(ws_proc, f'A{data_start}', f'{get_column_letter(len(headers))}{data_end}')
        # Fondo verde en all_valid cuando = 1
        if 'all_valid' in col_letter and n_rows > 0:
            from openpyxl.formatting.rule import CellIsRule
            av_letter = col_letter['all_valid']
            av_range = f'{av_letter}{data_start}:{av_letter}{data_end}'
            ws_proc.conditional_formatting.add(av_range,
                CellIsRule(operator='equal', formula=['1'],
                           fill=PatternFill('solid', start_color='C6EFCE', end_color='C6EFCE'),
                           font=Font(bold=True, color='006100')))

        _add_table(ws_proc, 'tbl_proc', 3, data_end, len(headers))
    proc_info = {
        'sheet': '03_SCADA_Procesado',
        'data_start': data_start,
        'data_end': data_end,
        'col_letter': col_letter,
        'headers': headers,
    }

    # ============================================================
    # 04_PVsyst_Simulacion
    # ============================================================
    ws_pv = wb.create_sheet('04_PVsyst_Simulacion')
    _explain(ws_pv,
        "Salida de la simulacion PVsyst (proc_pvsyst), alimentada con el meteo medido on-site "
        "(via meteo_export). Referencia de energia esperada contra la que se compara la real.",
        ncols=min(len(pvsyst_df.columns), 8))
    pv_cols = [c for c in pvsyst_df.columns if not pvsyst_df[c].isna().all()]
    pvsyst_df = pvsyst_df[pv_cols]
    _hdr(ws_pv, pv_cols, 3)
    for r_idx, row_vals in enumerate(pvsyst_df.itertuples(index=False, name=None), start=4):
        for c_idx, v in enumerate(row_vals, 1):
            _put_val(ws_pv, r_idx, c_idx, v,
                     fmt=('yyyy-mm-dd hh:mm' if pv_cols[c_idx-1] == 'Date' else None))
    ws_pv.freeze_panes = 'B4'
    _autosize(ws_pv)
    pv_col_letter = {h: get_column_letter(i+1) for i, h in enumerate(pv_cols)}
    pv_data_start = 4
    pv_data_end = 4 + len(pvsyst_df) - 1
    _add_table(ws_pv, 'tbl_pv', 3, pv_data_end, len(pv_cols))

    # ============================================================
    # 05_Comparacion_Interval (formulas)
    # ============================================================
    ws_cmp = wb.create_sheet('05_Comparacion_Interval')
    _explain(ws_cmp,
        "Comparacion a la resolucion del test (1 h si PVsyst es horario o aggregate_by_hour=True; "
        "si no, la mas fina comun). El detalle a 15 min esta en 03_SCADA_Procesado." + chr(10) +
        " - E_medida_kWh      = PROMEDIO(E_GRID del bucket en 03) * dt_h   (formula vinculada)" + chr(10) +
        " - E_esperada_kWh    = E_Grid PVsyst de esa hora (enlace a 04) * dt_h   (formula vinculada)" + chr(10) +
        " - valid             = MIN(all_valid de los sub-intervalos): la hora solo es valida si lo son sus 4 cuartos" + chr(10) +
        " - E_garantizada_kWh = E_esperada * EPI * Degradation * Availability   (formula viva)" + chr(10) +
        " - delta_kWh         = E_medida - E_garantizada   (formula viva)" + chr(10) +
        "Coloreado: gris = no valida (valid=0), verde = E_medida >= E_garantizada, rojo = E_medida < E_garantizada.",
        ncols=8)
    cmp_headers = ['Date', 'POA_avg_filt', 'E_medida_kWh', 'E_esperada_PVsyst_kWh',
                   'E_garantizada_kWh', 'delta_kWh', 'valid', 'POA_esperada', 'Hora']
    _hdr(ws_cmp, cmp_headers, 3)
    cmp_col = {h: get_column_letter(i+1) for i, h in enumerate(cmp_headers)}
    cmp_data_start = 4

    pv_egrid = 'E_Grid' if 'E_Grid' in pv_cols else next(
        (c for c in pv_cols if c != 'Date' and pd.api.types.is_numeric_dtype(pvsyst_df[c])), None
    )
    egrid_col = next((c for c in headers if c.upper().startswith('E_GRID')), None)

    # --- Agregacion a la resolucion del test (mismo criterio que pr_runner) ---
    # El detalle de 03 sigue a 15 min; la comparacion se hace a la resolucion del
    # test: 1 h si PVsyst es horario (o aggregate_by_hour=True). E_GRID y POA se
    # promedian, all_valid se agrega con MIN (la hora solo cuenta si lo son sus 4
    # cuartos) y E_esperada usa el valor horario real -> la delta cuadra con pr_runner.
    _sc_diffs  = scada_proc_df['Date'].diff().dropna()
    _scada_res = _sc_diffs.mode().iloc[0] if not _sc_diffs.empty else pd.Timedelta(minutes=15)
    _pv_diffs  = pvsyst_df['Date'].diff().dropna()
    _pv_res    = _pv_diffs.mode().iloc[0] if not _pv_diffs.empty else pd.Timedelta(hours=1)
    if cfg.get('aggregate_by_hour'):
        _target = max(pd.Timedelta(hours=1), _scada_res, _pv_res)
    else:
        _target = max(_scada_res, _pv_res)
    _dt_h_cmp = _target.total_seconds() / 3600.0
    _freq = pd.tseries.frequencies.to_offset(_target)
    print(f"  Comparacion @ {_target} (dt_h={_dt_h_cmp})")

    # Vinculacion por formulas: cada bucket (a la resolucion del test) es un rango
    # contiguo de filas de 03 (scada_proc esta ordenado por Date, ya con time_shift).
    _date_l  = proc_info['col_letter']['Date']
    _egrid_l = proc_info['col_letter'].get(egrid_col) if egrid_col else None
    _poa_l   = proc_info['col_letter'].get('POA_avg_filt')
    _valid_l = proc_info['col_letter'].get('all_valid')
    _psheet  = proc_info['sheet']

    _bkeys = scada_proc_df['Date'].dt.floor(_freq)

    # buckets contiguos por la fecha REAL (sin shift): [fila_03_inicio, fila_03_fin]
    _buckets = []
    _prev = None
    for _off, _bk in enumerate(_bkeys):
        _pr = data_start + _off
        if _prev is None or _bk != _prev[0]:
            _buckets.append([_bk, _pr, _pr]); _prev = _buckets[-1]
        else:
            _prev[2] = _pr

    r_cmp = cmp_data_start
    for _bk, _r0, _r1 in _buckets:
        _A = f"A{r_cmp}"
        # Date = fecha de 03 + time_shift/24  (dinamico)
        ws_cmp.cell(row=r_cmp, column=1,
                    value=f"='{_psheet}'!{_date_l}{_r0}").number_format = 'yyyy-mm-dd hh:mm'
        # POA medida = PROMEDIO del bucket en 03
        if _poa_l:
            ws_cmp.cell(row=r_cmp, column=2,
                        value=f"=IFERROR(AVERAGE('{_psheet}'!{_poa_l}{_r0}:{_poa_l}{_r1}),\"\")").number_format = '0.00'
        # E_medida = PROMEDIO(E_GRID del bucket) * dt_h
        if _egrid_l:
            ws_cmp.cell(row=r_cmp, column=3,
                        value=f"=IFERROR(AVERAGE('{_psheet}'!{_egrid_l}{_r0}:{_egrid_l}{_r1}),0)*{_dt_h_cmp}").number_format = '0.000'
        else:
            ws_cmp.cell(row=r_cmp, column=3, value=0).number_format = '0.000'
        # E_esperada = E_Grid PVsyst de la hora (segun fecha YA desplazada) * dt_h  (dinamico)
        if pv_egrid:
            ws_cmp.cell(row=r_cmp, column=4,
                        value=f"=SUMPRODUCT((INT(tbl_pv[Date]*24)=INT({_A}*24))*tbl_pv[{pv_egrid}])*{_dt_h_cmp}").number_format = '0.000'
        else:
            ws_cmp.cell(row=r_cmp, column=4, value=0).number_format = '0.000'
        # E_garantizada = E_esperada * EPI * Deg * Avail
        ee_ref = f'{cmp_col["E_esperada_PVsyst_kWh"]}{r_cmp}'
        ws_cmp.cell(row=r_cmp, column=5,
                    value=f'={ee_ref}*{REF_EPI_GUAR}*{REF_DEGRADATION}*{REF_AVAILABILITY}').number_format = '0.000'
        # delta = medida - garantizada
        em_ref = f'{cmp_col["E_medida_kWh"]}{r_cmp}'
        eg_ref = f'{cmp_col["E_garantizada_kWh"]}{r_cmp}'
        ws_cmp.cell(row=r_cmp, column=6, value=f'={em_ref}-{eg_ref}').number_format = '0.000'
        # valid = MIN(all_valid del bucket)
        if _valid_l:
            ws_cmp.cell(row=r_cmp, column=7,
                        value=f"=MIN('{_psheet}'!{_valid_l}{_r0}:{_valid_l}{_r1})")
        else:
            ws_cmp.cell(row=r_cmp, column=7, value=0)
        # POA_esperada = POA PVsyst de la hora (fecha desplazada)  (dinamico)
        if 'POA' in pv_col_letter:
            ws_cmp.cell(row=r_cmp, column=8,
                        value=f"=IFERROR(SUMPRODUCT((INT(tbl_pv[Date]*24)=INT({_A}*24))*tbl_pv[POA]),\"\")").number_format = '0.00'
        else:
            ws_cmp.cell(row=r_cmp, column=8, value="").number_format = '0.00'
        # Hora del dia (de la fecha desplazada) para el perfil horario
        ws_cmp.cell(row=r_cmp, column=9, value=f"=HOUR({_A})").number_format = '0'
        r_cmp += 1

    n_cmp = len(_buckets)
    ws_cmp.freeze_panes = 'B4'
    _autosize(ws_cmp)
    cmp_data_end = cmp_data_start + n_cmp - 1
    _add_table(ws_cmp, 'tbl_cmp', 3, cmp_data_end, len(cmp_headers))

    # --- Formato condicional: coloreado de NULL + filas por estado ---
    if n_rows > 0:
        last_col_letter = get_column_letter(len(cmp_headers))
        data_range = f'A{cmp_data_start}:{last_col_letter}{cmp_data_end}'

        # 1) Pintar celdas con texto 'NULL'/'DISCARDED' etc.
        _apply_status_cf(ws_cmp, f'A{cmp_data_start}', f'{last_col_letter}{cmp_data_end}')

        # 2) Coloreado de filas por status (orden importa: las primeras reglas ganan)
        from openpyxl.formatting.rule import FormulaRule
        # 2a) Gris si valid = 0 (intervalo no contado)
        ws_cmp.conditional_formatting.add(data_range,
            FormulaRule(formula=[f'$G{cmp_data_start}=0'],
                        fill=PatternFill('solid', start_color='E7E6E6', end_color='E7E6E6'),
                        stopIfTrue=True))
        # 2b) Verde si E_medida >= E_garantizada (intervalo valido y por encima de garantia)
        ws_cmp.conditional_formatting.add(data_range,
            FormulaRule(formula=[f'AND(ISNUMBER($C{cmp_data_start}),ISNUMBER($E{cmp_data_start}),$C{cmp_data_start}>=$E{cmp_data_start})'],
                        fill=PatternFill('solid', start_color='E2EFDA', end_color='E2EFDA'),
                        stopIfTrue=True))
        # 2c) Rojo si E_medida < E_garantizada
        ws_cmp.conditional_formatting.add(data_range,
            FormulaRule(formula=[f'AND(ISNUMBER($C{cmp_data_start}),ISNUMBER($E{cmp_data_start}),$C{cmp_data_start}<$E{cmp_data_start})'],
                        fill=PatternFill('solid', start_color='FCE4D6', end_color='FCE4D6'),
                        stopIfTrue=True))

    cmp_info = {
        'sheet': '05_Comparacion_Interval',
        'data_start': cmp_data_start,
        'data_end': cmp_data_end,
        'col_letter': cmp_col,
    }

    # ============================================================
    # 06_Resumen_Diario (formulas SUMPRODUCT por dia)
    # ============================================================
    ws_d = wb.create_sheet('06_Resumen_Diario')
    _explain(ws_d,
        "Agregado por dia (SUMPRODUCT sobre 05). Columnas '_valid' usan solo intervalos con valid=1. "
        "EPI = (E_medida_valid - E_garantizada_valid)/E_garantizada_valid (delta % sobre lo valido). "
        "POA medido/esperado = promedio SOLO en intervalos validos.",
        ncols=12)
    d_headers = ['Fecha', 'E_medida_kWh', 'E_esperada_kWh', 'E_garantizada_kWh', 'delta_kWh',
                 'E_medida_valid_kWh', 'E_garantizada_valid_kWh', 'delta_valid_kWh', 'EPI',
                 'medida_POA', 'esperada_POA', 'intervalos_validos', 'EPI_pos', 'EPI_neg']
    _hdr(ws_d, d_headers, 3)
    unique_days = pd.Series(scada_proc_df['Date'].dt.date.unique()).sort_values().tolist()
    d_data_start = 4

    for i, day in enumerate(unique_days):
        r = d_data_start + i
        ws_d.cell(row=r, column=1, value=day).number_format = 'yyyy-mm-dd'
        dref = f"$A{r}*1"
        # Totales del dia (todos los intervalos)
        ws_d.cell(row=r, column=2, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[E_medida_kWh])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=3, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[E_esperada_PVsyst_kWh])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=4, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[E_garantizada_kWh])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=5, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[delta_kWh])').number_format = '#,##0.0'
        # Solo intervalos validos
        ws_d.cell(row=r, column=6, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[valid]*tbl_cmp[E_medida_kWh])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=7, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[valid]*tbl_cmp[E_garantizada_kWh])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=8, value=f'=F{r}-G{r}').number_format = '#,##0.0'
        ws_d.cell(row=r, column=9, value=f'=IFERROR((F{r}-G{r})/G{r}*100,"")').number_format = '0.00 "%"'
        # POA medido y esperado: promedio SOLO en validos
        ws_d.cell(row=r, column=10,
                  value=f'=IFERROR(AVERAGEIFS(tbl_cmp[POA_avg_filt],tbl_cmp[Date],">="&{dref},tbl_cmp[Date],"<"&({dref}+1),tbl_cmp[valid],1),"")'
                  ).number_format = '0.00'
        ws_d.cell(row=r, column=11,
                  value=f'=IFERROR(AVERAGEIFS(tbl_cmp[POA_esperada],tbl_cmp[Date],">="&{dref},tbl_cmp[Date],"<"&({dref}+1),tbl_cmp[valid],1),"")'
                  ).number_format = '0.00'
        ws_d.cell(row=r, column=12, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[valid])').number_format = '0'
        ws_d.cell(row=r, column=13, value=f'=IF(I{r}="","",IF(I{r}>=0,I{r},NA()))').number_format = '0.00'
        ws_d.cell(row=r, column=14, value=f'=IF(I{r}="","",IF(I{r}<0,I{r},NA()))').number_format = '0.00'

    ws_d.freeze_panes = 'A4'
    _autosize(ws_d, max_w=18)
    _last_d = d_data_start + len(unique_days) - 1
    _add_table(ws_d, 'tbl_daily', 3, _last_d, len(d_headers))

    # Fila de TOTALES
    _tot = _last_d + 1
    ws_d.cell(row=_tot, column=1, value='TOTAL').font = Font(bold=True)
    for _ci, _cl in [(2, 'B'), (3, 'C'), (4, 'D'), (5, 'E'), (6, 'F'), (7, 'G'), (12, 'L')]:
        _cc = ws_d.cell(row=_tot, column=_ci, value=f'=SUM({_cl}{d_data_start}:{_cl}{_last_d})')
        _cc.number_format = '#,##0' if _ci == 12 else '#,##0.0'
        _cc.font = Font(bold=True)
    _cc = ws_d.cell(row=_tot, column=8, value=f'=F{_tot}-G{_tot}'); _cc.number_format = '#,##0.0'; _cc.font = Font(bold=True)
    _cc = ws_d.cell(row=_tot, column=9, value=f'=IFERROR((F{_tot}-G{_tot})/G{_tot}*100,"")'); _cc.number_format = '0.00 "%"'; _cc.font = Font(bold=True)

    # ============================================================
    # ============================================================
    # 00_Resumen (relleno final)
    # ============================================================
    em_sum_f = '=SUMIFS(tbl_cmp[E_medida_kWh], tbl_cmp[valid], 1)'
    ee_sum_f = '=SUMIFS(tbl_cmp[E_esperada_PVsyst_kWh], tbl_cmp[valid], 1)'
    eg_sum_f = '=SUMIFS(tbl_cmp[E_garantizada_kWh], tbl_cmp[valid], 1)'

    # B-rows positions (info_rows en 00_Resumen, empieza en row=3):
    #   row 3: Planta
    #   row 4: Periodo
    #   row 5: AC
    #   row 6: DC
    #   row 7: EPI
    #   row 8: Degradation
    #   row 9: Availability
    #   row 10: (empty)
    #   row 11: Energia medida
    #   row 12: Energia esperada
    #   row 13: Energia garantizada
    #   row 14: Delta kWh = B11 - B13
    #   row 15: Delta % = (B11-B13)/B13
    #   row 16: Resultado = IF(B11>=B13,"PASS","FAIL")
    info_rows = [
        ('Planta',                          f"{cfg['plant_name']}  ({plant_code})", None),
        ('Periodo',                         period_str, None),
        ('Capacidad AC (kW)',               cfg['ac_capacity_kw'], '#,##0'),
        ('Capacidad DC (kW)',               cfg['dc_capacity_kw'], '#,##0'),
        ('EPI garantizada',                 f'={REF_EPI_GUAR}', '0.0000'),
        ('Degradation factor',              f'={REF_DEGRADATION}', '0.0000'),
        ('Availability factor',             f'={REF_AVAILABILITY}', '0.0000'),
        ('', '', None),
        ('Energia medida (kWh)',            em_sum_f, '#,##0.0'),
        ('Energia esperada PVsyst (kWh)',   ee_sum_f, '#,##0.0'),
        ('Energia garantizada (kWh)',       eg_sum_f, '#,##0.0'),
        ('Delta (kWh)',                     '=B11-B13', '#,##0.0'),
        ('Delta (%)',                       '=IFERROR((B11-B13)/B13*100,"")', '0.00 "%"'),
        ('Ratio (medida/garantizada)',      '=IFERROR(B11/B13,"")', '0.0000'),
        ('EPI realizado',                   '=IFERROR(B11/B12,"")', '0.0000'),
        ('EPI garantizada (EPI*Deg*Avail)', f'={REF_EPI_GUAR}*{REF_DEGRADATION}*{REF_AVAILABILITY}', '0.0000'),
        ('Resultado',                       '=IF(B11>=B13,"PASS","FAIL")', None),
        ('', '', None),
        ('Intervalos 15 min (crudo)',       '=COUNTA(tbl_proc[Date])', '#,##0'),
        ('Intervalos 15 min validos',       '=COUNTIF(tbl_proc[all_valid],1)', '#,##0'),
        ('Intervalos de test',              '=COUNTA(tbl_cmp[Date])', '#,##0'),
        ('Intervalos de test validos',      '=COUNTIF(tbl_cmp[valid],1)', '#,##0'),
        ('Horas operativas validas',        f'=COUNTIF(tbl_cmp[valid],1)*{_dt_h_cmp}', '#,##0.0'),
    ]
    for i, (k, v, fmt) in enumerate(info_rows):
        r = 3 + i
        ws_sum.cell(row=r, column=1, value=k).font = Font(bold=bool(k))
        if v != '':
            cell = ws_sum.cell(row=r, column=2, value=v)
            if fmt:
                cell.number_format = fmt
            if k == 'Resultado':
                cell.font = _KPI_FONT
                from openpyxl.formatting.rule import FormulaRule
                ws_sum.conditional_formatting.add(f'B{r}',
                    FormulaRule(formula=[f'$B${r}="PASS"'], fill=_KPI_FILL_OK))
                ws_sum.conditional_formatting.add(f'B{r}',
                    FormulaRule(formula=[f'$B${r}="FAIL"'], fill=_KPI_FILL_KO))
    ws_sum.column_dimensions['A'].width = 32
    ws_sum.column_dimensions['B'].width = 28

    # ============================================================
    # 07_Grafica (energia diaria valida)
    # ============================================================
    from openpyxl.chart import LineChart, BarChart, Reference
    from openpyxl.chart.shapes import GraphicalProperties
    ws_g = wb.create_sheet('07_Grafica')
    _explain(ws_g, "Energia valida y EPI. Perfil horario = promedio por hora del dia sobre el rango de "
                   "fechas (edita Desde/Hasta). EPI = (medida_v - garant_v)/garant_v. Barras: azul>=0, rojo<0.",
             ncols=10)
    _AZUL = '5B9BD5'
    _ROJO = 'D99694'
    if len(unique_days) > 0:
        _d06 = '06_Resumen_Diario'
        cats_d = Reference(wb[_d06], min_col=1, min_row=d_data_start, max_row=_last_d)

        # 1) Lineas diarias: medida_valid (F=6) vs garantizada_valid (G=7)
        ch1 = LineChart()
        ch1.title = "Energia diaria valida - medida vs garantizada"
        ch1.style = 12; ch1.y_axis.title = 'kWh'; ch1.x_axis.title = 'Fecha'
        ch1.height = 9; ch1.width = 26
        for _c in (6, 7):
            ch1.add_data(Reference(wb[_d06], min_col=_c, min_row=3, max_row=_last_d), titles_from_data=True)
        ch1.set_categories(cats_d)
        ws_g.add_chart(ch1, "B2")

        # 2) EPI diario (barras azul/rojo, escala -100..100)
        ch2 = BarChart(); ch2.type = "col"
        ch2.title = "EPI diario (%)"
        ch2.y_axis.title = '%'; ch2.x_axis.title = 'Fecha'
        ch2.height = 9; ch2.width = 26; ch2.overlap = 100; ch2.gapWidth = 60
        ch2.add_data(Reference(wb[_d06], min_col=13, min_row=3, max_row=_last_d), titles_from_data=True)
        ch2.add_data(Reference(wb[_d06], min_col=14, min_row=3, max_row=_last_d), titles_from_data=True)
        ch2.set_categories(cats_d)
        ch2.y_axis.scaling.min = -100; ch2.y_axis.scaling.max = 100
        ch2.x_axis.tickLblPos = "low"
        ch2.series[0].graphicalProperties = GraphicalProperties(solidFill=_AZUL)
        ch2.series[1].graphicalProperties = GraphicalProperties(solidFill=_ROJO)
        ws_g.add_chart(ch2, "B20")

        # ---- Filtro de fechas (editable) en T2/U2 y T3/U3 ----
        _dmin = pd.Timestamp(scada_proc_df['Date'].min()).to_pydatetime()
        _dmax = pd.Timestamp(scada_proc_df['Date'].max()).to_pydatetime()
        ws_g.cell(row=2, column=20, value='Desde:').font = Font(bold=True)
        ws_g.cell(row=2, column=21, value=_dmin).number_format = 'yyyy-mm-dd'
        ws_g.cell(row=3, column=20, value='Hasta:').font = Font(bold=True)
        ws_g.cell(row=3, column=21, value=_dmax).number_format = 'yyyy-mm-dd'
        _DES = "$U$2"; _HAS = "$U$3"

        # ---- Tabla perfil horario (0..23): promedio sobre el filtro ----
        _hr0 = 5
        for _j, _hname in enumerate(['Hora', 'medida_valid', 'garantizada_valid', 'EPI', 'EPI_pos', 'EPI_neg']):
            ws_g.cell(row=_hr0, column=20 + _j, value=_hname).font = Font(bold=True)
        for _h in range(24):
            _r = _hr0 + 1 + _h
            ws_g.cell(row=_r, column=20, value=_h).number_format = '0'
            _flt = (f'tbl_cmp[Hora],T{_r},tbl_cmp[valid],1,'
                    f'tbl_cmp[Date],">="&{_DES},tbl_cmp[Date],"<"&({_HAS}+1)')
            ws_g.cell(row=_r, column=21, value=f'=IFERROR(AVERAGEIFS(tbl_cmp[E_medida_kWh],{_flt}),"")').number_format = '#,##0.000'
            ws_g.cell(row=_r, column=22, value=f'=IFERROR(AVERAGEIFS(tbl_cmp[E_garantizada_kWh],{_flt}),"")').number_format = '#,##0.000'
            ws_g.cell(row=_r, column=23, value=f'=IFERROR((U{_r}-V{_r})/V{_r}*100,"")').number_format = '0.00 "%"'
            ws_g.cell(row=_r, column=24, value=f'=IF(W{_r}="","",IF(W{_r}>=0,W{_r},NA()))').number_format = '0.00'
            ws_g.cell(row=_r, column=25, value=f'=IF(W{_r}="","",IF(W{_r}<0,W{_r},NA()))').number_format = '0.00'
        _hr_end = _hr0 + 24
        cats_h = Reference(ws_g, min_col=20, min_row=_hr0 + 1, max_row=_hr_end)

        # 3) Perfil horario combinado: barras EPI (azul/rojo, eje ppal) + lineas energia (eje 2o)
        bar_h = BarChart(); bar_h.type = "col"
        bar_h.title = "Perfil horario (promedio de dias filtrados) - EPI y energia"
        bar_h.height = 11; bar_h.width = 28; bar_h.overlap = 100; bar_h.gapWidth = 60
        bar_h.add_data(Reference(ws_g, min_col=24, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        bar_h.add_data(Reference(ws_g, min_col=25, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        bar_h.set_categories(cats_h)
        bar_h.y_axis.title = 'EPI %'
        bar_h.y_axis.scaling.min = -100; bar_h.y_axis.scaling.max = 100
        bar_h.x_axis.title = 'Hora del dia'
        bar_h.series[0].graphicalProperties = GraphicalProperties(solidFill=_AZUL)
        bar_h.series[1].graphicalProperties = GraphicalProperties(solidFill=_ROJO)

        line_h = LineChart()
        line_h.add_data(Reference(ws_g, min_col=21, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        line_h.add_data(Reference(ws_g, min_col=22, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        line_h.set_categories(cats_h)
        line_h.y_axis.axId = 200
        line_h.y_axis.title = 'kWh'
        line_h.y_axis.crosses = "max"
        bar_h += line_h
        ws_g.add_chart(bar_h, "B38")
    else:
        ws_g.cell(row=3, column=1, value="(sin datos para graficar)").font = Font(italic=True, color='6F6F6F')

    # ============================================================
    # 08_Outliers_POA (subset)
    # ============================================================
    ws_o = wb.create_sheet('08_Outliers')
    _explain(ws_o,
        f"Solo los intervalos donde al menos un sensor POA fue descartado por desviarse mas del "
        f"{cfg.get('poa_sensor_max_dev', 0.04)*100:.0f}% de la media de los POAs del mismo intervalo.",
        ncols=8)
    if outliers_df.empty:
        ws_o.cell(row=3, column=1, value="(No hubo descartes de outliers POA/GHI/DHI/REF en este periodo)").font = Font(italic=True, color='6F6F6F')
    else:
        out_cols = list(outliers_df.columns)
        _hdr(ws_o, out_cols, 3)
        for r_idx, row_vals in enumerate(outliers_df.itertuples(index=False, name=None), start=4):
            for c_idx, v in enumerate(row_vals, 1):
                _put_val(ws_o, r_idx, c_idx, v,
                         fmt=('yyyy-mm-dd hh:mm' if out_cols[c_idx-1] == 'Date' else None))
        ws_o.freeze_panes = 'B4'
        _autosize(ws_o)
        _apply_status_cf(ws_o, 'A4', f'{get_column_letter(len(out_cols))}{4 + len(outliers_df) - 1}')

    # ============================================================
    # Orden final
    # ============================================================
    new_order = ['00_Resumen', '01_Config_Contrato', '02_SCADA_Raw',
                 '03_SCADA_Procesado', '04_PVsyst_Simulacion',
                 '05_Comparacion_Interval', '06_Resumen_Diario',
                 '07_Grafica', '08_Outliers']
    wb._sheets = [wb[n] for n in new_order if n in wb.sheetnames]

    return wb


In [ ]:
def _norm_plant_codes(v):
    """Normaliza plant_codes a lista (una o varias). None/'' = todas.
    Acepta lista/tupla, JSON array o cadena separada por comas."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


plant_list = _norm_plant_codes(plant_codes)
if not plant_list:
    plant_list = [r['plant_code'] for r in spark.sql(
        "SELECT DISTINCT plant_code FROM scada_proc ORDER BY plant_code").collect()]
print(f"Plantas a procesar ({len(plant_list)}): {plant_list}")


def process_plant(plant_code):
    _ps = period_start
    _pe = period_end

    # ---------------------------------------------------------------------------
    # Pipeline: leer datos, construir el libro, subirlo
    # ---------------------------------------------------------------------------
    print(f"Cargando configuración de {plant_code}...")
    cfg = load_plant_config(plant_code)

    # Determinar periodo a partir de la INTERSECCIÓN de scada_proc y proc_pvsyst
    # (no podemos hacer trazabilidad donde no haya simulación PVsyst contra la que comparar)
    sc_row = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM scada_proc
        WHERE plant_code = '{plant_code}'
    """).first()
    pv_row = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM proc_pvsyst
        WHERE plant_code = '{plant_code}'
    """).first()

    if sc_row['mn'] is None:
        raise ValueError(f"No hay datos en scada_proc para {plant_code} — ejecuta primero scada_ingest + scada_proc.")
    if pv_row['mn'] is None:
        raise ValueError(
            f"No hay datos en proc_pvsyst para {plant_code}. "
            "Ejecuta antes el PVsyst CLI con la salida de meteo_export y luego csv_ingest."
        )

    print(f"  scada_proc  disponible: {sc_row['mn']} -> {sc_row['mx']}")
    print(f"  proc_pvsyst disponible: {pv_row['mn']} -> {pv_row['mx']}")

    intersect_start = max(pd.Timestamp(sc_row['mn']), pd.Timestamp(pv_row['mn']))
    intersect_end   = min(pd.Timestamp(sc_row['mx']), pd.Timestamp(pv_row['mx']))
    if intersect_end < intersect_start:
        raise ValueError(
            f"Sin solape entre scada_proc y proc_pvsyst para {plant_code}: "
            f"scada=[{sc_row['mn']}..{sc_row['mx']}], pvsyst=[{pv_row['mn']}..{pv_row['mx']}]"
        )

    # Normaliza lo que haya escrito el usuario (acepta ISO YYYY-MM-DD; vacío = no fijado)
    _req_start = pd.Timestamp(_ps) if str(_ps).strip() else None
    _req_end   = pd.Timestamp(_pe)   if str(_pe).strip()   else None

    if _req_start is not None or _req_end is not None:
        # PRIORIDAD: si fijas _ps/_pe, mandan sobre last_month_only
        _ps_dt = _req_start if _req_start is not None else intersect_start
        _pe_dt   = _req_end   if _req_end   is not None else intersect_end

        # Comprobar disponibilidad y recortar al solape real de datos
        if _ps_dt < intersect_start:
            print(f"  AVISO: inicio pedido {_ps_dt} < disponible {intersect_start}; se recorta al disponible.")
            _ps_dt = intersect_start
        if _pe_dt > intersect_end:
            print(f"  AVISO: fin pedido {_pe_dt} > disponible {intersect_end}; se recorta al disponible.")
            _pe_dt = intersect_end
        if _pe_dt < _ps_dt:
            raise ValueError(
                f"El periodo pedido [{_ps} .. {_pe}] no se solapa con los datos "
                f"disponibles [{intersect_start} .. {intersect_end}]."
            )

        _ps = _ps_dt.isoformat()
        _pe   = _pe_dt.isoformat()
        print(f"Modo 'periodo manual' (_ps/_pe tienen prioridad): {_ps} -> {_pe}")
    elif last_month_only:
        _pe_dt   = intersect_end
        _ps_dt = max(intersect_start, _pe_dt - timedelta(days=30))
        _ps = _ps_dt.isoformat()
        _pe   = _pe_dt.isoformat()
        print(f"Modo 'último mes' (sobre la intersección de datos): {_ps} -> {_pe}")
    else:
        _ps = intersect_start.isoformat()
        _pe   = intersect_end.isoformat()
        print(f"Modo 'rango completo' (toda la intersección de datos): {_ps} -> {_pe}")

    _ps_f = pd.Timestamp(_ps).strftime('%Y%m%d')
    _pe_f   = pd.Timestamp(_pe).strftime('%Y%m%d')
    period_str = f"{pd.Timestamp(_ps).strftime('%d/%m/%Y')} a {pd.Timestamp(_pe).strftime('%d/%m/%Y')}"
    print(f"Periodo final: {period_str}")

    # --- Aviso temprano: comprobar que las 3 tablas tienen datos en el periodo ANTES de hacer el trabajo pesado ---
    print("\nComprobando disponibilidad de datos en el periodo...")
    for tbl in ('scada_raw', 'scada_proc', 'proc_pvsyst'):
        n = spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM {tbl}
            WHERE plant_code = '{plant_code}'
              AND Date BETWEEN TIMESTAMP'{_ps}' AND TIMESTAMP'{_pe}'
        """).first()['n']
        print(f"  {tbl:12s}: {n:,} filas")
        if n == 0:
            # Pista útil: ¿qué rango tiene esa tabla globalmente para la planta?
            rg = spark.sql(f"""
                SELECT MIN(Date) AS mn, MAX(Date) AS mx
                FROM {tbl}
                WHERE plant_code = '{plant_code}'
            """).first()
            if rg['mn'] is None:
                extra = "la tabla está completamente vacía para esta planta."
                if tbl == 'proc_pvsyst':
                    extra += " Ejecuta el PVsyst CLI con la salida de meteo_export y luego csv_ingest."
            else:
                extra = f"la tabla tiene datos solo entre {rg['mn']} y {rg['mx']}."
            raise ValueError(
                f"{tbl} vacío para {plant_code} en el periodo {_ps} -> {_pe}. "
                f"Pista: {extra}"
            )


    def _read_table(table, plant_code, _ps, _pe):
        sql = f"""
            SELECT *
            FROM {table}
            WHERE plant_code = '{plant_code}'
              AND Date BETWEEN TIMESTAMP'{_ps}' AND TIMESTAMP'{_pe}'
            ORDER BY Date
        """
        return spark.sql(sql).toPandas()


    print("\nLeyendo datos...")
    scada_raw  = _read_table('scada_raw',  plant_code, _ps, _pe).drop(columns=['source_file', 'ingested_at', 'plant_code'], errors='ignore')
    scada_proc = _read_table('scada_proc', plant_code, _ps, _pe).drop(columns=['source_file', 'ingested_at', 'processed_at', 'plant_code'], errors='ignore')
    pvsyst     = _read_table('proc_pvsyst', plant_code, _ps, _pe).drop(columns=['source_file', 'ingested_at', 'plant_code'], errors='ignore')

    print(f"  scada_raw : {len(scada_raw):,} filas")
    print(f"  scada_proc: {len(scada_proc):,} filas")
    print(f"  pvsyst    : {len(pvsyst):,} filas")

    # Detectar dt_h del SCADA
    diffs = scada_raw['Date'].diff().dropna()
    dt_h = float(diffs.mode().iloc[0].total_seconds() / 3600) if not diffs.empty else 0.25
    print(f"  dt_h      : {dt_h} horas/intervalo")
    # ---------------------------------------------------------------------------
    # Subset de outliers (para la hoja 08; el resto va con fórmulas)
    # ---------------------------------------------------------------------------
    # Outliers de todas las familias de sensores (POA, GHI, DHI, REF).
    # Se ignoran columnas todo-nulos (sensor no instalado en esta planta).
    _OUTLIER_FAMILIES = [
        ('POA_', 'POA_avg_filt'),
        ('GHI_', 'GHI_avg_filt'),
        ('DHI_', 'DHI_avg_filt'),
        ('REF_', 'REF_avg_filt'),
    ]
    _all_out_idx  = set()
    _all_keep_cols = []
    for _pfx, _avg in _OUTLIER_FAMILIES:
        _raw  = sorted([c for c in scada_proc.columns
                        if c.upper().startswith(_pfx)
                        and not c.endswith('_filt') and not c.endswith('_flag')
                        and c != _avg])
        _filt = sorted([c for c in scada_proc.columns
                        if c.upper().startswith(_pfx)
                        and c.endswith('_filt') and c != _avg])
        # Descartar columnas enteramente nulas (sensor no existe en esta planta)
        _raw  = [c for c in _raw  if not scada_proc[c].isna().all()]
        _filt = [c for c in _filt if not scada_proc[c].isna().all()]
        if not _raw or not _filt:
            continue
        _raw_num  = scada_proc[_raw].apply(pd.to_numeric, errors='coerce')
        _filt_num = scada_proc[_filt].apply(pd.to_numeric, errors='coerce')
        _mask = _filt_num.isna().any(axis=1) & (_raw_num.fillna(0).abs().sum(axis=1) > 0)
        _all_out_idx.update(scada_proc.index[_mask].tolist())
        _all_keep_cols += _raw + _filt
        if _avg in scada_proc.columns and not scada_proc[_avg].isna().all():
            _all_keep_cols.append(_avg)
    if _all_out_idx:
        outliers_df = scada_proc.loc[sorted(_all_out_idx)].copy()
        _seen = set(); _keep = ['Date']
        for _c in _all_keep_cols:
            if _c not in _seen and _c in outliers_df.columns:
                _keep.append(_c); _seen.add(_c)
        outliers_df = outliers_df[_keep]
    else:
        outliers_df = pd.DataFrame()

    print(f"  Filas con descartes (POA/GHI/DHI/REF): {len(outliers_df):,}")
    # ---------------------------------------------------------------------------
    # Construir Excel y subirlo
    # ---------------------------------------------------------------------------
    wb = build_workbook(plant_code, period_str, cfg, dt_h,
                        scada_raw_df=scada_raw,
                        scada_proc_df=scada_proc,
                        pvsyst_df=pvsyst,
                        outliers_df=outliers_df)

    buf = io.BytesIO()
    wb.save(buf)
    content_bytes = buf.getvalue()
    print(f"\nExcel generado: {len(content_bytes)/1024:.1f} KB")

    # Nombre del fichero
    fname = f"{plant_code}_PR_Traceability_{_ps_f}-{_pe_f}.xlsx"

    # 1) Audit local
    audit_dir = f"{LAKEHOUSE_FILES}/{local_audit_root.strip('/')}/{plant_code}"
    Path(audit_dir).mkdir(parents=True, exist_ok=True)
    audit_path = f"{audit_dir}/{fname}"
    with open(audit_path, 'wb') as f:
        f.write(content_bytes)
    print(f"  audit local: {audit_path.replace('/lakehouse/default/', '')}")

    # 2) Upload SharePoint vía Graph
    sharepoint_url = None
    try:
        print("\nAutenticando contra Microsoft Graph...")
        _token  = get_graph_token()
        _site_id, _drive_id = resolve_site_and_drive(_token)
        dest_relpath = f"{sp_traceability_path.strip('/')}/{plant_code}/{fname}"
        info = upload_to_sharepoint(_token, _site_id, _drive_id, dest_relpath, content_bytes)
        sharepoint_url = info.get('webUrl')
        print(f"  SharePoint:  {sharepoint_url}")
    except Exception as _e:
        print(f"  WARN: upload a SharePoint fallido (se conserva la copia local): {_e}")

    return {
        'plant_code':      plant_code,
        'period_start':    _ps_f,
        'period_end':      _pe_f,
        'rows_scada_raw':  int(len(scada_raw)),
        'rows_scada_proc': int(len(scada_proc)),
        'rows_pvsyst':     int(len(pvsyst)),
        'rows_outliers':   int(len(outliers_df)),
        'local_audit':     audit_path.replace('/lakehouse/default/', ''),
        'sharepoint':      sharepoint_url,
    }



In [ ]:
import notebookutils
results = []
for _pc in plant_list:
    try:
        results.append({'status': 'ok', **process_plant(_pc)})
    except Exception as _e:
        results.append({'plant_code': _pc, 'status': 'failed', 'error': str(_e)[:300]})
        print(f"  FALLO {_pc}: {_e}")

_summary = {
    'plants_processed': len(results),
    'ok':     sum(1 for r in results if r.get('status') == 'ok'),
    'failed': sum(1 for r in results if r.get('status') == 'failed'),
    'results': results,
}
print('\n' + json.dumps(_summary, indent=2, default=str))
notebookutils.notebook.exit(json.dumps(_summary, default=str))